# 00 — Preprocessing

Preprocessing sekarang dijalankan lewat modul `src/preprocessing`
(`make preprocess`), bukan dari notebook ini, supaya training (notebook
`01`–`04`) dan serving (`src/prediction`) memakai kode fitur yang sama persis:
isi NaN (median per jam periode train), MinMaxScaler per stasiun, kalender
sin/cos, dan akumulasi hujan DAS per stasiun (rata-rata sel terpilih,
dijumlah 1/3/6/12/24 jam; scaler log1p + MinMax).

Urutan sebelum training:

    make raw-preprocess-incremental   # station_features_hourly
    make rainfall-survey              # hujan semua sel, 1 musim hujan
    make rainfall-rank                # ranking sel per stasiun
    make rainfall-bq                  # histori hujan sel terpilih
    make preprocess                   # preprocessed_data/metadata/scalers_hourly

Notebook ini hanya memeriksa hasilnya di BigQuery.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

import json
import pandas as pd

PROJECT_ID = "thesis-506605"
DATASET    = "tables"
LOCATION   = "US"

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

meta_row = client.query(f"""
    SELECT payload, saved_at
    FROM `{PROJECT_ID}.{DATASET}.preprocessing_metadata_hourly`
    ORDER BY saved_at DESC
    LIMIT 1
""").to_dataframe().iloc[0]
metadata = json.loads(meta_row["payload"])

print(f"Metadata disimpan : {meta_row['saved_at']}")
print(f"Stasiun           : {metadata['station_cols']}")
print(f"Fitur dasar       : {len(metadata['base_feature_cols'])} (kalender: {metadata['calendar_feature_cols']})")
print(f"Akumulasi hujan   : {metadata['rain_accum_hours']} jam")
for station, cols in metadata["feature_cols_by_station"].items():
    cells = metadata["rain_cells_by_station"].get(station, [])
    print(f"  {station:<20} {len(cols):>2} fitur | {len(cells)} sel hujan: {cells}")

In [ ]:
client.query(f"""
    SELECT split, COUNT(*) AS n_rows, MIN(datetime) AS start, MAX(datetime) AS `end`
    FROM `{PROJECT_ID}.{DATASET}.preprocessed_data_hourly`
    GROUP BY split
    ORDER BY start
""").to_dataframe()